> **Student version** -- exercise cells only recall the instruction: you write the code yourself, without any imposed skeleton. A worked example on an analogous case always precedes this kind of exercise. Cells marked **Question** have no automatic correction: run the code, observe, and answer in writing. The solution notebook is available on the course webpage.


# PAC-Bayes 4 -- Learning the weights of a forest by minimizing a bound

A PAC-Bayesian bound holds for all posteriors simultaneously, so we can **minimize it** over the weights $\rho$ of the voters: the resulting vote comes with its own risk certificate
(*self-bounding* learning, lecture notes, Sections 6 and 7). We work with a random forest whose trees are evaluated on their out-of-bag (OOB) samples, and compare three posteriors:
the uniform one (standard random forest), the minimizer of the **first-order** bound (Thiemann et al., 2017) and the minimizer of the **tandem** bound (Masegosa et al., 2020).
The last part (Master level) minimizes a PAC-Bayesian C-bound directly (in the spirit of Viallard et al., 2021).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.datasets import make_classification, load_digits, load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

rng = np.random.RandomState(0)

class BaggedTrees:
    def __init__(self, n_estimators=100, max_features="sqrt", max_depth=None, bootstrap_frac=0.5, random_state=0):
        self.n_estimators, self.max_features, self.max_depth = n_estimators, max_features, max_depth
        self.bootstrap_frac, self.random_state = bootstrap_frac, random_state
    def fit(self, X, y):
        r = np.random.RandomState(self.random_state)
        m = len(y); k = int(self.bootstrap_frac * m)
        self.trees_, oob = [], []
        for t in range(self.n_estimators):
            idx = r.randint(0, m, k)
            mask = np.ones(m, bool); mask[idx] = False
            self.trees_.append(DecisionTreeClassifier(max_features=self.max_features, max_depth=self.max_depth,
                                                      random_state=r.randint(1 << 30)).fit(X[idx], y[idx]))
            oob.append(mask)
        self.oob_ = np.array(oob); return self
    def votes(self, X):
        return np.array([t.predict(X) for t in self.trees_])

def oob_stats(bt, X, y):
    # OOB losses (first order), tandem losses and disagreements on OOB intersections
    V = bt.votes(X); O = bt.oob_.astype(float); Err = (V != y).astype(float)
    Eo = Err * O
    L = Eo.sum(1) / O.sum(1); n1 = int(O.sum(1).min())
    both = O @ O.T
    Lt = (Eo @ Eo.T) / np.maximum(both, 1); n2 = int(both.min())
    D = np.zeros_like(both)
    for t in range(len(V)):
        D[t] = ((V[t] != V) * O[t] * O).sum(1)
    D = D / np.maximum(both, 1)
    return L, n1, Lt, n2, D

def kl_bin(q, p):
    p = min(max(p, 1e-12), 1 - 1e-12)
    t1 = q * np.log(q / p) if q > 0 else 0.0
    t2 = (1 - q) * np.log((1 - q) / (1 - p)) if q < 1 else 0.0
    return t1 + t2

def kl_inv_upper(q, eps, tol=1e-9):
    lo, hi = q, 1.0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if kl_bin(q, mid) > eps: hi = mid
        else: lo = mid
    return hi

def kl_inv_lower(q, eps, tol=1e-9):
    lo, hi = 0.0, q
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if kl_bin(q, mid) > eps: lo = mid
        else: hi = mid
    return lo

def kl_div(q, p):
    mask = q > 0
    return np.sum(q[mask] * np.log(q[mask] / p[mask]))

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def fo_bound(rho, pi, L, n, delta=0.05):
    return min(1, 2 * kl_inv_upper(rho @ L, (kl_div(rho, pi) + np.log(2 * np.sqrt(n) / delta)) / n))

def tnd_bound(rho, pi, Lt, n, delta=0.05):
    return min(1, 4 * kl_inv_upper(rho @ Lt @ rho, (2 * kl_div(rho, pi) + np.log(2 * np.sqrt(n) / delta)) / n))

def mv_risk(V, y, rho):
    return np.mean(np.where(rho @ V >= 0, 1, -1) != y)

In [ ]:
Xd, yd = load_digits(return_X_y=True)
yd = np.where(yd >= 5, 1, -1)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0)
bt = BaggedTrees(100, random_state=0).fit(Xtr, ytr)
L, n1, Lt, n2, D = oob_stats(bt, Xtr, ytr)
Vte = bt.votes(Xte)
pi = np.ones(100) / 100
print(f"m={len(ytr)}, n1={n1}, n2={n2}")
print(f"uniform: test MV {mv_risk(Vte, yte, pi):.3f}  FO bound {fo_bound(pi, pi, L, n1):.3f}  TND bound {tnd_bound(pi, pi, Lt, n2):.3f}")


## I. Minimizing the first-order bound

The PAC-Bayes-$\lambda$ bound gives $R(G_\rho)\leq \frac{\rho^\top\hat L}{1-\lambda/2}+\frac{\mathrm{KL}(\rho\|\pi)+\ln\frac{2\sqrt{n}}{\delta}}{\lambda(1-\lambda/2)n}$. For fixed $\lambda$, the minimizer is the Gibbs posterior
$\rho_t\propto\pi_t e^{-\lambda n\hat L_t}$; for fixed $\rho$, $\lambda$ has the closed form $\lambda^\star=2/(\sqrt{2n\rho^\top\hat L/(\mathrm{KL}+\ln\frac{2\sqrt n}\delta)+1}+1)$. We alternate (Algorithm 3 of the lecture notes).


In [ ]:
def optimize_fo(pi, L, n, delta=0.05, n_iter=200):
    lam = 1.0
    for _ in range(n_iter):
        rho = softmax(np.log(pi) - lam * n * L)
        c = kl_div(rho, pi) + np.log(2 * np.sqrt(n) / delta)
        lam_new = 2 / (np.sqrt(2 * n * (rho @ L) / c + 1) + 1)
        if abs(lam_new - lam) < 1e-10:
            break
        lam = lam_new
    return rho

rho_fo = optimize_fo(pi, L, n1)
print(f"FO-optimized: test MV {mv_risk(Vte, yte, rho_fo):.3f}  FO bound {fo_bound(rho_fo, pi, L, n1):.3f}  "
      f"TND bound {tnd_bound(rho_fo, pi, Lt, n2):.3f}  KL {kl_div(rho_fo, pi):.2f}")
print("effective number of trees 1/sum(rho^2):", 1 / np.sum(rho_fo**2))
plt.bar(range(100), np.sort(rho_fo)[::-1]); plt.axhline(0.01, color="C3", label="uniform")
plt.xlabel("trees (sorted)"); plt.ylabel("rho"); plt.legend(); plt.show()

$$ $$

**Question 1:** Compare the FO certificate and the test risk of the vote for the uniform and the FO-optimized posterior. How many trees effectively vote after optimization? Why does the vote get worse while the certificate improves?

$$ $$


## II. Minimizing the tandem bound

The tandem bound with the PAC-Bayes-$\lambda$ relaxation reads
$$\mathcal B_{\mathrm{TND}}(\rho,\lambda)=4\Big[\frac{\rho^\top\hat{\mathbf L}\rho}{1-\lambda/2}+\frac{2\mathrm{KL}(\rho\|\pi)+\ln\frac{2\sqrt{n_2}}{\delta}}{\lambda(1-\lambda/2)n_2}\Big].$$
There is no closed form in $\rho$ anymore. We parametrize $\rho=\mathrm{softmax}(\theta)$ and use a gradient-based solver. The chain rule through the softmax gives
$\nabla_\theta=\rho\odot(g-\rho^\top g)$ where $g=\nabla_\rho$. Below is the same machinery applied to the **first-order** objective for a fixed $\lambda$: we check that we recover the Gibbs posterior.


In [ ]:
def fo_objective(theta, pi, L, n, lam, delta=0.05):
    rho = softmax(theta)
    c = np.log(2 * np.sqrt(n) / delta)
    val = (rho @ L) / (1 - lam / 2) + (kl_div(rho, pi) + c) / (lam * (1 - lam / 2) * n)
    g = L / (1 - lam / 2) + (np.log(np.maximum(rho, 1e-300) / pi) + 1) / (lam * (1 - lam / 2) * n)
    return val, rho * (g - rho @ g)

lam = 0.3
res = minimize(fo_objective, np.log(pi), args=(pi, L, n1, lam), jac=True, method="L-BFGS-B")
rho_grad = softmax(res.x)
rho_closed = softmax(np.log(pi) - lam * n1 * L)
print("max |difference| with the Gibbs posterior:", np.abs(rho_grad - rho_closed).max())


### Exercise 1

Write `optimize_tnd(pi, Lt, n2, delta)` which alternates (i) a minimization in $\theta$ of $\mathcal B_{\mathrm{TND}}(\mathrm{softmax}(\theta),\lambda)$ with L-BFGS (write the gradient: $\nabla_\rho=\frac{2\hat{\mathbf L}\rho}{1-\lambda/2}+\frac{2(\ln(\rho/\pi)+1)}{\lambda(1-\lambda/2)n_2}$) and
(ii) the closed-form update $\lambda^\star=2/(\sqrt{2n_2\,\rho^\top\hat{\mathbf L}\rho/(2\mathrm{KL}+\ln\frac{2\sqrt{n_2}}\delta)+1}+1)$, for 20 rounds at most.
Report the test risk of the vote, the TND certificate (with `tnd_bound`) and the effective number of trees, and compare with the uniform and FO posteriors.


In [ ]:
# optimize_tnd: alternate L-BFGS on softmax logits (tandem lambda-objective, analytic gradient)
# and the closed-form lambda; then test MV, TND bound, effective number of trees.


$$ $$

**Question 2:** Why does the tandem objective keep many trees? Look at the term $\rho^\top\hat{\mathbf L}\rho$: which pairs of trees does it penalize?

$$ $$


## III. Comparison on several datasets

Let us run the three posteriors on four datasets, with 5 random splits each.


In [ ]:
def load_all():
    out = {}
    X, y = load_breast_cancer(return_X_y=True); out["breast cancer"] = (X, 2 * y - 1)
    X, y = load_digits(return_X_y=True); out["digits"] = (X, np.where(y >= 5, 1, -1))
    X, y = make_moons(2000, noise=0.3, random_state=0); out["moons"] = (X, 2 * y - 1)
    X, y = make_classification(3000, 20, n_informative=8, flip_y=0.05, random_state=2); out["simulated"] = (X, 2 * y - 1)
    return out

def run(X, y, seed):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=seed)
    f = BaggedTrees(100, random_state=seed).fit(Xa, ya)
    L_, n1_, Lt_, n2_, _ = oob_stats(f, Xa, ya)
    V_ = f.votes(Xb); p_ = np.ones(100) / 100
    out = []
    for r in [p_, optimize_fo(p_, L_, n1_), optimize_tnd(p_, Lt_, n2_)]:
        out.append([mv_risk(V_, yb, r), fo_bound(r, p_, L_, n1_), tnd_bound(r, p_, Lt_, n2_)])
    return np.array(out)

table = {}
for name, (X, y) in load_all().items():
    table[name] = np.mean([run(X, y, s) for s in range(5)], axis=0)
    print(name)
    for lab, row in zip(["uniform", "FO-opt", "TND-opt"], table[name]):
        print(f"   {lab:8s} test MV {row[0]:.3f}   FO bound {row[1]:.3f}   TND bound {row[2]:.3f}")

$$ $$

**Question 3:** Summarize: which posterior gives the best vote? which gives the best certificate? Is there a dataset on which the FO-optimized forest is competitive, and why?

$$ $$


## IV. (Master) Direct minimization of a PAC-Bayesian C-bound

The PAC-Bayesian C-bound combines an upper bound on the Gibbs risk and a lower bound on the disagreement (union bound, $\delta/2$ each):
$$R(B_\rho)\leq 1-\frac{(1-2\bar r)^2}{1-2\underline d},\quad \bar r=\overline{\mathrm{kl}}^{-1}\Big(\rho^\top\hat L,\tfrac{\mathrm{KL}+\ln\frac{4\sqrt{n_1}}{\delta}}{n_1}\Big),\quad
\underline d=\underline{\mathrm{kl}}^{-1}\Big(\rho^\top\hat D\rho,\tfrac{2\mathrm{KL}+\ln\frac{4\sqrt{n_2}}{\delta}}{n_2}\Big).$$
Here is the function computing it.


In [ ]:
def cbound_pb(rho, pi, L, n1, D, n2, delta=0.05):
    KL = kl_div(rho, pi)
    r = kl_inv_upper(rho @ L, (KL + np.log(4 * np.sqrt(n1) / delta)) / n1)
    d = kl_inv_lower(rho @ D @ rho, (2 * KL + np.log(4 * np.sqrt(n2) / delta)) / n2)
    return 1.0 if r >= 0.5 else min(1.0, 1 - (1 - 2 * r)**2 / (1 - 2 * d))

for name, r in [("uniform", pi), ("FO", rho_fo), ("TND", rho_tnd)]:
    print(f"{name:8s} PAC-Bayesian C-bound {cbound_pb(r, pi, L, n1, D, n2):.3f}")


### Exercise 2

Minimize `cbound_pb` over $\rho=\mathrm{softmax}(\theta)$, starting from the uniform posterior, with `scipy.optimize.minimize(method="Powell")` or L-BFGS with numerical gradients (the function is not smooth everywhere, a derivative-free method is safer; limit the number of evaluations to keep the run under a minute).
Compare the certificate and the test risk of the vote with the three previous posteriors. Is the self-bounding C-bound learner a good compromise?


In [ ]:
# Minimize the PAC-Bayesian C-bound over softmax(theta) from the uniform posterior (Powell,
# limited evaluations); compare certificate and test risk with uniform / FO / TND.



## Conclusion

Bound minimization is a principled way to learn the weights of a vote *and* to certify it. The order of the bound matters: first-order bounds lead to
concentrated posteriors and poor votes, second-order bounds (tandem, C-bound) preserve the diversity of the ensemble. This is the core of the project of this part of the course.
